In [2]:
import geopandas as gpd
from shapely.ops import linemerge

gdf = gpd.read_file("kathmandu_river.geojson").to_crs(32645)  # UTM 45N, metres

# keep only the main rivers: drop short fragments
gdf["len"] = gdf.length
gdf = gdf[gdf["len"] > 1500]            # raise to 3000 for even fewer streams

# merge touching segments into longer lines
merged = linemerge(list(gdf.geometry.explode(index_parts=False)))
lines = gpd.GeoDataFrame(geometry=list(merged.geoms) if hasattr(merged, "geoms") else [merged],
                         crs=32645)

# simplify: removes jitter, keeps overall shape
lines["geometry"] = lines.geometry.simplify(40, preserve_topology=True)  # metres; 20 = detailed, 80 = very simple

# optional: smooth corners (Chaikin)
def chaikin(geom, n=2):
    from shapely.geometry import LineString
    pts = list(geom.coords)
    for _ in range(n):
        new = [pts[0]]
        for a, b in zip(pts, pts[1:]):
            new += [(0.75*a[0]+0.25*b[0], 0.75*a[1]+0.25*b[1]),
                    (0.25*a[0]+0.75*b[0], 0.25*a[1]+0.75*b[1])]
        new.append(pts[-1])
        pts = new
    return LineString(pts)

lines["geometry"] = lines.geometry.apply(chaikin)

lines.to_crs(4326).to_file("kathmandu_river_simple.geojson", driver="GeoJSON")